# Elliptical Magnet Array Optimization - Example Usage

This notebook demonstrates using the `ellipse_opt` package for GPU-accelerated permanent magnet array optimization.

**This is an example usage notebook.** Parameters shown are for demonstration - adjust for your specific design requirements.

**Package Features:**
- 3-phase hybrid perturbation (exploration → transition → exploitation)
- S-curve temperature scaling
- Ben-Ameur auto-calibration
- REFLECTION bounds handling
- Bolt hole avoidance constraints


## 1. Imports

In [1]:
import numpy as np
import cupy as cp
from datetime import datetime

from ellipse_opt import (
    field_calculator, geometry, sampling, generator,
    perturbation, constraints, cost, optimizer,
    calibration, visualization
)
GRFC = field_calculator
print("Imports OK")

Imports OK


## 2. Load Field Data

In [2]:
B = np.load('Precalc_field_450mm_grid_1mm_step_Br1.48_demag0.9836_20260701_220818.npz')
Precalc_Bxyz_value = B['Bxyzvalue'].astype(np.float32)
Ref_grid = B['Ref_grid']
step_size = float(B['step_size'])
GRFC.load_field(Precalc_Bxyz_value, Ref_grid, step_size)

✓ Field loaded to GPU: (91125000, 3) (1.09 GB)
  Grid: 450x450x450, step=1.0mm


## 3. Parameters (EXACT from original)

In [3]:
inmm = 1e-3
halbach_magnet_size = 6.35 * inmm  # 6.35mm cube magnets
layer_clearance = 1.524 * inmm     # Shimstock thickness
sheet_thickness = halbach_magnet_size + layer_clearance

# Geometry parameters (user-defined)
n_magnets_q1 = 2170      # Magnets in half-ellipse (reflected to full)
z_min = 0.0              # Minimum z (half-z symmetry, z >= 0)
z_max = 200 * inmm       # Maximum z (200mm half-length)
rx_inner = 150 * inmm    # Ellipse semi-axis x (150mm)
ry_inner = 125 * inmm    # Ellipse semi-axis y (125mm)
gap = 40 * inmm          # Radial gap for magnets (40mm)
magnet_size = halbach_magnet_size
n_sample_points = 500    # DSV sample points

# DSV (Design Spherical Volume) parameters
dsv_radius = max(rx_inner, ry_inner) * 0.57  # 57% of larger bore dimension
sphere_radius = dsv_radius
z_min_dsv = -(z_max - halbach_magnet_size)
z_max_dsv = z_max - halbach_magnet_size

# Cost function weights
weights = {
    'field_strength': 50000,      # Weight for maximizing field
    'uniformity': 50000*20,       # Weight for field homogeneity (1,000,000)
    'z_barrier': 5.0,             # Soft barrier at z bounds
    'overlap_barrier': 5.0,       # Soft barrier for magnet overlap
    'radius_barrier': 5.0,        # Soft barrier at radial bounds
    'bolt_barrier': 1.0           # Soft barrier for bolt holes
}

def calculate_step_divisors(magnet_size, z_min, z_max, layer_thickness, layer_clearance,
                            r_inner, r_outer, exploration_scale=1.0, phi_resolution_deg=10.0):
    d_min = magnet_size * cp.sqrt(2)
    sheet = layer_thickness + layer_clearance
    r_avg = (r_inner + r_outer) / 2
    z_range = z_max - z_min
    r_range = r_outer - r_inner
    return {
        'z': max(1, int(np.round(z_range / (sheet * exploration_scale)))),
        'r': max(1, int(np.round(r_range / (d_min * exploration_scale)))),
        'theta': max(1, int(np.round((2*np.pi) / ((d_min / r_avg) * exploration_scale)))),
        'phi': max(1, int(np.round(np.pi / (np.radians(phi_resolution_deg) * exploration_scale))))
    }

step_divisors = calculate_step_divisors(
    halbach_magnet_size, z_min, z_max, halbach_magnet_size, layer_clearance,
    (rx_inner + ry_inner) / 2, (rx_inner + ry_inner) / 2 + gap, 4.5, 10.0
)

# SA parameters
cooling_rate = 0.999         # Temperature decay rate
display_interval = 1000      # Print progress every N iterations
initial_temp = 1.0           # Starting temperature (calibration will override)
n_iterations = 200000        # Total iterations
explore_exploit_pct = 0.85   # Switch to transition phase at 85%
fine_tune_pct = 0.92         # Switch to fine-tune phase at 92%

print(f"Step divisors: {step_divisors}")
print(f"Design: {rx_inner*2000:.0f}x{ry_inner*2000:.0f}mm bore, {n_magnets_q1*4} magnets")


Step divisors: {'z': 6, 'r': 1, 'theta': 24, 'phi': 4}
Design: 300x250mm bore, 8680 magnets


## 4. Sample Points

In [4]:
sample_points = sampling._generate_sample_points_small_dsv(n_sample_points, sphere_radius, z_min_dsv, z_max_dsv)
print(f"Generated {len(sample_points)} DSV sample points")

Generated 500 DSV sample points


## 5. Generate Initial Config

In [5]:
initial_q1 = generator.generate_half_ellipse(n_magnets_q1, z_max, rx_inner, ry_inner, gap, halbach_magnet_size)
full_config = generator.reflect_to_half_ellipse(initial_q1)
print(f"Q1: {len(initial_q1)}, Full: {len(full_config)}")

Generating 2170 magnets in half-ellipse (y ≥ 0)...
θ range: [0, π] = [0°, 180°]
Layer range: 0 to 24
Generated 2170 magnets in half-ellipse
Q1: 2170, Full: 8680


## 6. Visualize

In [6]:
visualization.Plot_positions_plotly(full_config, sample_points)

## 7. Calibration (Optional)

In [7]:
# ======================================================================
# RUN CALIBRATION
# ======================================================================

T0, Tf, alpha, diag = calibration.calibrate_sa_rigorous(
    initial_config=initial_q1,
    z_min=z_min,
    z_max=z_max,
    rx_inner=rx_inner,
    ry_inner=ry_inner,
    gap=gap,
    magnet_size=halbach_magnet_size,
    sample_points=sample_points,
    weights=weights,
    step_divisors=step_divisors,
    Precalc_Bxyz_value=Precalc_Bxyz_value,
    Ref_grid=Ref_grid,
    step_size=step_size,
    cooling_rate=cooling_rate,
    n_iterations=100000,  # Temp value for sampling (actual set later)
    n_samples=200,
    chi_0=0.9,
    chi_f=0.01,
    moves_per_level=1,
    hot_temp=1e4,
    cold_ratio=0.01,
)

# Apply calibration
initial_temp = T0
# Apply calibration
cooling_rate = alpha
print(f"\n✓ Applied: initial_temp = {initial_temp:,.0f}, cooling_rate = {cooling_rate:.8f}")

# ======================================================================
# AUTO-CALCULATE ITERATIONS FOR TARGET FINAL TEMPERATURE
# ======================================================================
# Formula: n = log(T_final / T0) / log(cooling_rate)

T_final_target = 1  # Target: near-greedy (acceptance ~1-5%)

# OPTION: Override iteration count (set to None for auto-calculation)
n_iterations_override = 300000  # User requested 300k

if n_iterations_override is not None:
    n_iterations = n_iterations_override
    # Recalculate cooling rate to reach T_final in specified iterations
    cooling_rate = (T_final_target / T0) ** (1 / n_iterations)
    print(f"\n>>> OVERRIDE: Using {n_iterations:,} iterations")
    print(f">>> Adjusted cooling_rate = {cooling_rate:.10f}")
    n_iterations_needed = n_iterations
else:
    # Calculate required iterations from calibrated cooling rate
    n_iterations_needed = int(np.log(T_final_target / T0) / np.log(cooling_rate))

print(f"\n{'='*60}")
print(f"AUTO-ITERATION CALCULATION")
print(f"{'='*60}")
print(f"  T0 = {T0:,.0f}")
print(f"  T_final_target = {T_final_target}")
print(f"  cooling_rate = {cooling_rate}")
print(f"  n_iterations needed = {n_iterations_needed:,}")

# Set n_iterations from calibration (unless override is active)
if n_iterations_override is None:
    n_iterations = n_iterations_needed
    print(f"  → Using auto-calculated n_iterations = {n_iterations:,}")
else:
    print(f"  → Using OVERRIDE n_iterations = {n_iterations:,}")

# Verify final temperature
T_final_actual = T0 * (cooling_rate ** n_iterations)
print(f"  T_final_actual = {T_final_actual:.2f}")
print(f"{'='*60}")


SA RIGOROUS CALIBRATION

Base cost: 234,463.33

Step 2: Random-walk sampling 200 moves (hot)...
  93 uphill moves from 201 configs
  ΔE  min 0.297   median 240   mean 896   max 9.6e+03

Step 2b: Sampling at cold step size (ratio 0.01)...
  37 uphill moves,  median ΔE 324

Step 1: Per-term ΔE variation across sampled configs...
  field_strength       std 1.13e+03
  uniformity           std 1.59e+04
  z_barrier            std 17
  overlap_barrier      std 140
  radius_barrier       std 30.7
  bolt_barrier         std 3.13
  ⚠️  imbalance 5,091× — the small term is invisible to SA

Step 3: Solving χ(T₀)=0.9, χ(T_f)=0.01 ...
  T₀  = 7,082   (verify χ = 90.0%)
  T_f = 1.219   (verify χ = 1.00%)
  sanity: T₀ / median(ΔE) = 29.53   (want ~0.1–10)

Step 4: cooling
  n_iterations 100,000  /  moves_per_level 1  →  K = 100,000 cooling events
  α = 0.99991333
  check: T₀·α^K = 1.219  (should equal T_f)

initial_temp = 7,082.07
cooling_rate = 0.99991333

✓ Applied: initial_temp = 7,082, cooling_rat

## 8. Run Optimization

In [ ]:
best_q1, best_full, best_cost, history = optimizer.simulated_annealing_half_ellipse(
    initial_q1, z_min, z_max, rx_inner, ry_inner, gap,
    halbach_magnet_size, sample_points, weights,
    Precalc_Bxyz_value, Ref_grid, step_size, step_divisors,
    initial_temp, cooling_rate, n_iterations, display_interval,
    explore_exploit_pct=explore_exploit_pct, fine_tune_pct=fine_tune_pct
)
print("Done!")

Starting simulated annealing (half-ellipse reflected ellipse)...
Optimizing 2170 magnets in Q1 -> 4340 total
Iter 1000/300000, T=6875.85, Cost=141942.12
  Accept: 96.6%, Improvements: 122, Invalid: 0
  TIMING: Perturb=62.4ms, Cost=47.2ms, Total=109.6ms
  Best: Field=0.68 mT, RMS=0.3218 mT, Homog=47.52%
  >>> Estimated remaining time: 9.2 hours (110.3 ms/iter avg)
Iter 2000/300000, T=6675.63, Cost=124620.79
  Accept: 96.6%, Improvements: 135, Invalid: 0
  TIMING: Perturb=62.0ms, Cost=46.7ms, Total=108.8ms
  Best: Field=0.70 mT, RMS=0.3137 mT, Homog=44.82%
Iter 3000/300000, T=6481.25, Cost=66018.11
  Accept: 98.2%, Improvements: 235, Invalid: 0
  TIMING: Perturb=60.9ms, Cost=46.5ms, Total=107.4ms
  Best: Field=0.56 mT, RMS=0.1840 mT, Homog=32.96%
Iter 4000/300000, T=6292.52, Cost=65816.82
  Accept: 96.3%, Improvements: 253, Invalid: 0
  TIMING: Perturb=61.1ms, Cost=46.9ms, Total=108.0ms
  Best: Field=0.59 mT, RMS=0.1587 mT, Homog=26.94%
  >>> Checkpoint saved: iter 5000, 0.73 mT, 22.97%


## 9. Results

In [ ]:
visualization.plot_cost_progression(history)

B_field, B_mag = GRFC.calc_field(best_full, sample_points)
avg_field = float(cp.mean(B_mag))
std_field = float(cp.std(B_mag))
print(f"Field: {avg_field:.2f} mT, Homogeneity: {std_field/avg_field*100:.2f}%")

## 10. Save

In [ ]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
np.savez(f"ellipse_design_{ts}.npz",
    q1_config=cp.asnumpy(best_q1), full_config=cp.asnumpy(best_full),
    cost=float(best_cost), field_mT=avg_field
)
print(f"Saved!")